### Lesson 3: Agentic Search

In [33]:
# !pip install tavily-python
# !pip install bs4
# !pip install duckduckgo_search
# !pip install ddgs

In [6]:
import os 
from dotenv import load_dotenv

load_dotenv()

gemini_api_key = os.getenv('GEMINI_API_KEY')
tavily_api_key = os.getenv('TAVILY_API_KEY')

In [8]:
from tavily import TavilyClient
client = TavilyClient(api_key=tavily_api_key)

In [10]:
result = client.search("What is in Nvidia's new Blackwell GPU?",
                       include_answer=True)
result['answer']

'The Nvidia Blackwell GPU features advanced Tensor Cores, Transformer Engine, and Confidential Computing for AI. It supports up to 576 GPUs and offers significant performance improvements.'

#### Regular Search

In [23]:
city = "New Delhi"
query = f"""What is the weather in {city}?
Should I travel there today?
"weather.com"
"""

In [24]:
import requests
from bs4 import BeautifulSoup
from duckduckgo_search import DDGS
import re

ddg = DDGS()

def search(query, max_results=6):
    try:
        results = ddg.text(query, max_results=max_results)
        return [i['href'] for i in results]
    except Exception as e:
        print(f"returning previous results due to exception reaching ddg.")
        results = [ # cover case where DDG rate limits due to high deeplearning.ai volume
            "https://weather.com/weather/today/l/USCA0987:1:US",
            "https://weather.com/weather/hourbyhour/l/54f9d8baac32496f6b5497b4bf7a277c3e2e6cc5625de69680e6169e7e38e9a8",
        ]
        return results

for i in search(query):
    print(i)


/var/folders/70/3n6r904d4zb5zh68lfyslrt80000gn/T/ipykernel_23027/3233235035.py:6: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  ddg = DDGS()


https://forum.wordreference.com/threads/all-day-all-time-all-weather.3714880/
https://www.reddit.com/r/politics/?feedViewType=cardView
https://www.reddit.com/r/allthemods/comments/19b3l79/is_it_possible_to_control_weather/
https://www.reddit.com/r/valheim/comments/lu3f4n/weather_command/
https://www.reddit.com/r/weather/comments/16suejq/good_weather_websites_that_are_not_weathercom/
https://www.reddit.com/r/androidapps/comments/18chwfh/whats_the_best_weather_app_for_android/


In [25]:
def scrape_weather_info(url):
    if not url:
        return "Weather information could not be found."
    
    headers = {'User-Agent': 'Mozilla/5.0'}
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return "Failed to retrieve the webpage."

    soup = BeautifulSoup(response.text, 'html.parser')
    return soup

In [28]:
url = search(query)[1]
soup = scrape_weather_info(url)

print(f"Website: {url}\n\n")
print(str(soup.body)[:50000]) # limit long outputs

Website: https://www.reddit.com/r/politics/?feedViewType=cardView


<body>
<a class="button-medium px-[var(--rem14)] button-primary items-center justify-center button inline-flex" href="#main-content" id="shreddit-skip-link" rpl=""><span class="flex items-center justify-center">
<span class="flex items-center gap-xs">Skip to main content</span>
</span>
</a>
<script nonce="ZyqaZPx/Y9Lj2+bubSO79g==">
          window.shouldTrackTTC = !document.hidden;
        </script>
<shreddit-app app-name="web3x" class="overflow-visible pt-[var(--page-y-padding)] nd-visible" clienthash="/CWDZZI0V4RJRnm7" cluster="MvqI0hhL" comments-partial-ssr="" correlation-id="7b0c536c-6933-474f-a4fe-24e680ae46c8" country="IN" ctn="CSRF" devicetype="desktop" feed-correlation-id="6a1a6797-5e4b-41c5-9725-a6530b3808bf" loid="000000001zc4gsess9.2.1759818096167.Z0FBQUFBQm81TEZ3TWEyTzVvWHRlLXdoY1NDSVJIQ1ZxR1UtajVHR2ZIcjN2eEk2aTBiUUJsZDFXaWRtbkNQTUt4M3B2R0RZZ0Z0VlB3NEROS1JZMmJwVUdSdjBvMXVIckRZNXFUbVJ4eXVKMXVyR213QXVvZTI0Qk

In [29]:
# extract text
weather_data = []
for tag in soup.find_all(['h1', 'h2', 'h3', 'p']):
    text = tag.get_text(" ", strip=True)
    weather_data.append(text)

# combine all elements into a single string
weather_data = "\n".join(weather_data)

# remove all spaces from the combined text
weather_data = re.sub(r'\s+', ' ', weather_data)
    
print(f"Website: {url}\n\n")
print(weather_data)

Website: https://www.reddit.com/r/politics/?feedViewType=cardView


r/politics Community highlights Discussion Thread: 2025 US Government Shutdown, Day 6 Discussion Thread: US Supreme Court Hears Oral Argument in Two Cases on Attorney-Client Consultations During Trial and State Versus Federal Law in Lawsuits Anyone can view, post, and comment to this community Community Bookmarks Site Wide Rules See our hate speech rule here See our harm rule here . Manipulating comments and posts via group voting is against reddit TOS r/politics Rules General: No hateful speech See our hate speech rule here . General: No novelty accounts or bots "Novelty" and "gimmick" accounts or bots are not allowed on r/politics General: No spam or solicitation Polls and petitions are not allowed here. This rule is applied to, but not limited to, any link or text post that is attempting to solicit money, signatures, poll responses, volunteer hours, or sign ups from users. "Solicitation" can include advertising a su

#### Agentic Search

In [30]:
result = client.search(query,
                       include_answer=True)
data = result['results'][0]['content']
print(data)

{'location': {'name': 'New Delhi', 'region': 'Delhi', 'country': 'India', 'lat': 28.6, 'lon': 77.2, 'tz_id': 'Asia/Kolkata', 'localtime_epoch': 1759818325, 'localtime': '2025-10-07 11:55'}, 'current': {'last_updated_epoch': 1759817700, 'last_updated': '2025-10-07 11:45', 'temp_c': 27.2, 'temp_f': 81.0, 'is_day': 1, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/day/116.png', 'code': 1003}, 'wind_mph': 7.6, 'wind_kph': 12.2, 'wind_degree': 142, 'wind_dir': 'SE', 'pressure_mb': 1011.0, 'pressure_in': 29.85, 'precip_mm': 2.55, 'precip_in': 0.1, 'humidity': 66, 'cloud': 50, 'feelslike_c': 29.0, 'feelslike_f': 84.2, 'windchill_c': 26.6, 'windchill_f': 80.0, 'heatindex_c': 28.3, 'heatindex_f': 82.9, 'dewpoint_c': 19.4, 'dewpoint_f': 67.0, 'vis_km': 6.0, 'vis_miles': 3.0, 'uv': 6.6, 'gust_mph': 14.0, 'gust_kph': 22.6}}


In [31]:
import json
from pygments import highlight, lexers, formatters

# parse JSON
parsed_json = json.loads(data.replace("'", '"'))

# pretty print JSON with syntax highlighting
formatted_json = json.dumps(parsed_json, indent=4)
colorful_json = highlight(formatted_json,
                          lexers.JsonLexer(),
                          formatters.TerminalFormatter())

print(colorful_json)

{
    "location": {
        "name": "New Delhi",
        "region": "Delhi",
        "country": "India",
        "lat": 28.6,
        "lon": 77.2,
        "tz_id": "Asia/Kolkata",
        "localtime_epoch": 1759818325,
        "localtime": "2025-10-07 11:55"
    },
    "current": {
        "last_updated_epoch": 1759817700,
        "last_updated": "2025-10-07 11:45",
        "temp_c": 27.2,
        "temp_f": 81.0,
        "is_day": 1,
        "condition": {
            "text": "Partly cloudy",
            "icon": "//cdn.weatherapi.com/weather/64x64/day/116.png",
            "code": 1003
        },
        "wind_mph": 7.6,
        "wind_kph": 12.2,
        "wind_degree": 142,
        "wind_dir": "SE",
        "pressure_mb": 1011.0,
        "pressure_in": 29.85,
        "precip_mm": 2.55,
        "precip_in": 0.1,
        "humidity": 66,
        "cloud": 50,
        "feelslike_c": 29.0,
        "feelslike_f": 84.2,
        "windchill_c": 26.6,
        "windchill_f": 80.0,
        "heatinde